# Auditoria piloto de AIME

Objetivo: comprobar acceso, configuracion, splits, numero de filas, columnas y features de `disco-eth/AIME` sin descargar ni decodificar audio.

In [1]:
import os
import sys
from pathlib import Path
from importlib import metadata

# Cache local ignorada por git; se fija antes de importar librerias de Hugging Face.
cwd = Path.cwd()
repo_root = cwd if (cwd / ".git").exists() else cwd.parent
os.environ.setdefault("HF_HOME", str(repo_root / ".cache" / "huggingface"))

import datasets
from huggingface_hub import HfApi
import numpy as np
import pandas as pd

packages = ["datasets", "huggingface_hub", "numpy", "pandas", "torch", "torchaudio", "librosa"]
versions = {
    "python": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

versions

C:\Users\sergio\tfm\tfm-ai-music-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'python_executable': 'C:\\Users\\sergio\\tfm\\tfm-ai-music-detection\\.venv\\Scripts\\python.exe',
 'datasets': '5.0.0',
 'huggingface_hub': '1.21.0',
 'numpy': '2.4.6',
 'pandas': '3.0.3',
 'torch': '2.13.0',
 'torchaudio': 'NOT INSTALLED',
 'librosa': '0.11.0'}

## 1. Acceso y esquema

In [2]:
dataset_id = "disco-eth/AIME"

api = HfApi()
repo_info = api.dataset_info(dataset_id, files_metadata=False)

hub_summary = {
    "id": repo_info.id,
    "sha": repo_info.sha,
    "siblings_count": len(repo_info.siblings or []),
    "first_files": [s.rfilename for s in (repo_info.siblings or [])[:8]],
    "card_data_type": type(repo_info.cardData).__name__,
}

hub_summary

{'id': 'disco-eth/AIME',
 'sha': 'b84d4be5eda830b6eb714998569dba73530f2601',
 'siblings_count': 212,
 'first_files': ['.gitattributes',
  'README.md',
  'data/train-00000-of-00210.parquet',
  'data/train-00001-of-00210.parquet',
  'data/train-00002-of-00210.parquet',
  'data/train-00003-of-00210.parquet',
  'data/train-00004-of-00210.parquet',
  'data/train-00005-of-00210.parquet'],
 'card_data_type': 'DatasetCardData'}

In [3]:
datasets_api = {
    "version": datasets.__version__,
    "has_get_dataset_config_names": hasattr(datasets, "get_dataset_config_names"),
    "has_get_dataset_split_names": hasattr(datasets, "get_dataset_split_names"),
    "has_get_dataset_infos": hasattr(datasets, "get_dataset_infos"),
    "has_get_dataset_config_info": hasattr(datasets, "get_dataset_config_info"),
}

configs = datasets.get_dataset_config_names(dataset_id)
splits = datasets.get_dataset_split_names(dataset_id)
dataset_info = datasets.get_dataset_config_info(dataset_id, config_name="default")

columns = list(dataset_info.features.keys())
features = {name: repr(feature) for name, feature in dataset_info.features.items()}
split_rows = {name: split.num_examples for name, split in dataset_info.splits.items()}
split_num_bytes = {name: split.num_bytes for name, split in dataset_info.splits.items()}

schema_summary = {
    "configs": configs,
    "splits": splits,
    "split_rows": split_rows,
    "columns": columns,
    "features": features,
    "split_num_bytes": split_num_bytes,
    "download_size": dataset_info.download_size,
    "dataset_size": dataset_info.dataset_size,
}

datasets_api, schema_summary

({'version': '5.0.0',
  'has_get_dataset_config_names': True,
  'has_get_dataset_split_names': True,
  'has_get_dataset_infos': True,
  'has_get_dataset_config_info': True},
 {'configs': ['default'],
  'splits': ['train'],
  'split_rows': {'train': 6500},
  'columns': ['id', 'model', 'description', 'audio'],
  'features': {'id': "Value('string')",
   'model': "Value('string')",
   'description': "Value('string')",
   'audio': 'Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)'},
  'split_num_bytes': {'train': 62747721096.5},
  'download_size': 62281475287,
  'dataset_size': 62747721096.5})

In [4]:
observed = {
    "method": "HfApi.dataset_info + datasets.get_dataset_config_names/get_dataset_split_names/get_dataset_config_info",
    "audio_downloaded_or_decoded": False,
    "dataset_id": dataset_id,
    "repo_sha": hub_summary["sha"],
    "configs": configs,
    "splits": splits,
    "rows": split_rows,
    "columns": columns,
    "features": features,
    "warnings": [
        "No se ha llamado a load_dataset ni se ha accedido a ejemplos o a la columna audio.",
        "La feature audio aparece con decode=True por defecto; acceder a ejemplos podria descargar o decodificar audio.",
    ],
}

print("RESULTADOS OBSERVADOS - FASE 1")
for key, value in observed.items():
    print(f"{key}: {value}")

RESULTADOS OBSERVADOS - FASE 1
method: HfApi.dataset_info + datasets.get_dataset_config_names/get_dataset_split_names/get_dataset_config_info
audio_downloaded_or_decoded: False
dataset_id: disco-eth/AIME
repo_sha: b84d4be5eda830b6eb714998569dba73530f2601
configs: ['default']
splits: ['train']
rows: {'train': 6500}
columns: ['id', 'model', 'description', 'audio']
features: {'id': "Value('string')", 'model': "Value('string')", 'description': "Value('string')", 'audio': 'Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)'}
warnings: ['No se ha llamado a load_dataset ni se ha accedido a ejemplos o a la columna audio.', 'La feature audio aparece con decode=True por defecto; acceder a ejemplos podria descargar o decodificar audio.']


## 2. Distribucion y agrupacion de metadatos

In [5]:
import time

import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

revision = "b84d4be5eda830b6eb714998569dba73530f2601"
metadata_columns = ["id", "model", "description"]

repo_files = list(
    api.list_repo_tree(
        dataset_id,
        repo_type="dataset",
        revision=revision,
        path_in_repo="data",
        recursive=True,
        expand=True,
    )
)

parquet_files = sorted(
    [file for file in repo_files if getattr(file, "path", "").endswith(".parquet")],
    key=lambda file: file.path,
)
remote_parquet_size_bytes = sum(getattr(file, "size", 0) or 0 for file in parquet_files)
parquet_paths = [f"datasets/{dataset_id}@{revision}/{file.path}" for file in parquet_files]

parquet_summary = {
    "revision": revision,
    "parquet_file_count": len(parquet_files),
    "first_paths": [file.path for file in parquet_files[:8]],
    "remote_parquet_size_bytes": remote_parquet_size_bytes,
    "remote_parquet_size_gib": remote_parquet_size_bytes / 1024**3,
}

parquet_summary

{'revision': 'b84d4be5eda830b6eb714998569dba73530f2601',
 'parquet_file_count': 210,
 'first_paths': ['data/train-00000-of-00210.parquet',
  'data/train-00001-of-00210.parquet',
  'data/train-00002-of-00210.parquet',
  'data/train-00003-of-00210.parquet',
  'data/train-00004-of-00210.parquet',
  'data/train-00005-of-00210.parquet',
  'data/train-00006-of-00210.parquet',
  'data/train-00007-of-00210.parquet'],
 'remote_parquet_size_bytes': 62281475287,
 'remote_parquet_size_gib': 58.00414391513914}

In [6]:
def directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file())

cache_root = Path(os.environ["HF_HOME"])
cache_before_bytes = directory_size_bytes(cache_root)

hf_fs = HfFileSystem()
start = time.perf_counter()
metadata_table = pq.read_table(
    parquet_paths,
    filesystem=hf_fs,
    columns=metadata_columns,
)
metadata_read_seconds = time.perf_counter() - start
cache_after_bytes = directory_size_bytes(cache_root)
cache_delta_bytes = cache_after_bytes - cache_before_bytes

metadata_read_summary = {
    "metadata_read_seconds": metadata_read_seconds,
    "cache_before_bytes": cache_before_bytes,
    "cache_after_bytes": cache_after_bytes,
    "cache_delta_bytes": cache_delta_bytes,
    "table_columns": metadata_table.column_names,
    "table_rows": metadata_table.num_rows,
}

metadata_read_summary

{'metadata_read_seconds': 181.54404010018334,
 'cache_before_bytes': 8149,
 'cache_after_bytes': 8149,
 'cache_delta_bytes': 0,
 'table_columns': ['id', 'model', 'description'],
 'table_rows': 6500}

In [7]:
assert metadata_table.column_names == metadata_columns

metadata_df = metadata_table.to_pandas()
assert list(metadata_df.columns) == metadata_columns

df_summary = {
    "shape": metadata_df.shape,
    "columns": list(metadata_df.columns),
    "memory_bytes_deep": int(metadata_df.memory_usage(deep=True).sum()),
}

print(df_summary)
metadata_df.head(5)

{'shape': (6500, 3), 'columns': ['id', 'model', 'description'], 'memory_bytes_deep': 431441}


,id,model,description
0,04501,Udio,"ambient, blues, piano"
1,04502,Udio,"ambient, breakbeat, trance"
2,04503,Udio,"ambient, calm, happy"
3,04504,Udio,"ambient, calm, soft"
4,04505,Udio,"ambient, calm, space"


In [8]:
model_values = sorted(metadata_df["model"].unique().tolist())
model_counts = metadata_df["model"].value_counts().sort_index()
human_mask = metadata_df["model"].eq("MTG-Jamendo")
ai_mask = ~human_mask

metadata_df = metadata_df.assign(label=human_mask.map({True: 0, False: 1}))
label_counts = metadata_df["label"].value_counts().sort_index()
model_label_relation = (
    metadata_df.groupby("model", sort=True)["label"]
    .agg(["nunique", "min", "max", "count"])
)

mtg_only_label_0 = metadata_df.loc[human_mask, "label"].eq(0).all()
other_models_only_label_1 = metadata_df.loc[ai_mask, "label"].eq(1).all()

model_audit = {
    "unique_models": model_values,
    "n_unique_models": len(model_values),
    "model_counts": model_counts.to_dict(),
    "human_rows": int(human_mask.sum()),
    "ai_rows": int(ai_mask.sum()),
    "label_counts": label_counts.to_dict(),
    "mtg_only_label_0": bool(mtg_only_label_0),
    "other_models_only_label_1": bool(other_models_only_label_1),
}

model_audit, model_label_relation

({'unique_models': ['AudioLDM 2 Large',
   'AudioLDM 2 Music',
   'MTG-Jamendo',
   'MusicGen Large',
   'MusicGen Medium',
   'MusicGen Small',
   'Mustango',
   'Riffusion',
   'Stable Audio v1',
   'Stable Audio v2',
   'Suno v3',
   'Suno v3.5',
   'Udio'],
  'n_unique_models': 13,
  'model_counts': {'AudioLDM 2 Large': 500,
   'AudioLDM 2 Music': 500,
   'MTG-Jamendo': 500,
   'MusicGen Large': 500,
   'MusicGen Medium': 500,
   'MusicGen Small': 500,
   'Mustango': 500,
   'Riffusion': 500,
   'Stable Audio v1': 500,
   'Stable Audio v2': 500,
   'Suno v3': 500,
   'Suno v3.5': 500,
   'Udio': 500},
  'human_rows': 500,
  'ai_rows': 6000,
  'label_counts': {0: 500, 1: 6000},
  'mtg_only_label_0': True,
  'other_models_only_label_1': True},
                   nunique  min  max  count
 model                                     
 AudioLDM 2 Large        1    1    1    500
 AudioLDM 2 Music        1    1    1    500
 MTG-Jamendo             1    0    0    500
 MusicGen Large         

In [9]:
description_types = metadata_df["description"].map(lambda value: type(value).__name__).value_counts()
description_examples = metadata_df["description"].drop_duplicates().head(10).map(repr).tolist()

def infer_description_format(values: pd.Series) -> str:
    sample = values.dropna().astype(str).head(50)
    if sample.empty:
        return "sin valores no nulos"
    json_like = sample.str.match(r"^\s*\[.*\]\s*$|^\s*\{.*\}\s*$").mean()
    python_list_like = sample.str.match(r"^\s*\[.*\]\s*$").mean()
    comma_joined = sample.str.contains(",").mean()
    if json_like > 0.8:
        return "posible lista u objeto serializado como texto"
    if python_list_like > 0.8:
        return "posible representacion Python de lista como texto"
    if comma_joined > 0.8:
        return "texto con tags concatenados por comas"
    return "texto simple u otro formato"

description_group_sizes = metadata_df.groupby("description", dropna=False).size().sort_values()
description_size_distribution = description_group_sizes.value_counts().sort_index()
description_stats = {
    "python_types": description_types.to_dict(),
    "examples_repr": description_examples,
    "inferred_format": infer_description_format(metadata_df["description"]),
    "unique_descriptions": int(metadata_df["description"].nunique(dropna=False)),
    "group_size_min": int(description_group_sizes.min()),
    "group_size_max": int(description_group_sizes.max()),
    "group_size_mean": float(description_group_sizes.mean()),
    "group_size_median": float(description_group_sizes.median()),
    "group_size_distribution": description_size_distribution.to_dict(),
}

description_stats

{'python_types': {'str': 6500},
 'examples_repr': ["'ambient, blues, piano'",
  "'ambient, breakbeat, trance'",
  "'ambient, calm, happy'",
  "'ambient, calm, soft'",
  "'ambient, calm, space'",
  "'ambient, chillout, downtempo'",
  "'ambient, chillout, drums'",
  "'ambient, chillout, hiphop'",
  "'ambient, chillout, piano'",
  "'ambient, chillout, soundtrack'"],
 'inferred_format': 'texto con tags concatenados por comas',
 'unique_descriptions': 500,
 'group_size_min': 13,
 'group_size_max': 13,
 'group_size_mean': 13.0,
 'group_size_median': 13.0,
 'group_size_distribution': {13: 500}}

In [10]:
group_composition = (
    metadata_df.groupby("description", dropna=False)
    .agg(
        rows=("id", "size"),
        human_rows=("model", lambda values: int((values == "MTG-Jamendo").sum())),
        ai_rows=("model", lambda values: int((values != "MTG-Jamendo").sum())),
        unique_models=("model", "nunique"),
        models_present=("model", lambda values: sorted(set(values))),
    )
    .sort_index()
)

expected_model_count = len(model_values)
groups_with_1_human = int(group_composition["human_rows"].eq(1).sum())
groups_with_12_ai = int(group_composition["ai_rows"].eq(12).sum())
groups_with_13_rows = int(group_composition["rows"].eq(13).sum())
groups_with_all_models = int(group_composition["unique_models"].eq(expected_model_count).sum())
anomalous_groups_df = group_composition.loc[
    ~(
        group_composition["human_rows"].eq(1)
        & group_composition["ai_rows"].eq(12)
        & group_composition["rows"].eq(13)
        & group_composition["unique_models"].eq(expected_model_count)
    )
]

group_audit = {
    "groups": int(len(group_composition)),
    "groups_with_1_human": groups_with_1_human,
    "groups_with_12_ai": groups_with_12_ai,
    "groups_with_13_rows": groups_with_13_rows,
    "groups_with_all_observed_models": groups_with_all_models,
    "anomalous_groups": int(len(anomalous_groups_df)),
}

group_audit, anomalous_groups_df

({'groups': 500,
  'groups_with_1_human': 500,
  'groups_with_12_ai': 500,
  'groups_with_13_rows': 500,
  'groups_with_all_observed_models': 500,
  'anomalous_groups': 0},
 Empty DataFrame
 Columns: [rows, human_rows, ai_rows, unique_models, models_present]
 Index: [])

In [11]:
null_values = metadata_df[metadata_columns].isna().sum()
empty_strings = metadata_df[metadata_columns].apply(
    lambda series: series.astype("string").str.strip().eq("").sum()
)
duplicate_ids = int(metadata_df["id"].duplicated().sum())
unique_ids = int(metadata_df["id"].nunique(dropna=False))

integrity_audit = {
    "null_values": null_values.to_dict(),
    "null_values_total": int(null_values.sum()),
    "duplicate_ids": duplicate_ids,
    "unique_ids": unique_ids,
    "empty_strings": {column: int(value) for column, value in empty_strings.to_dict().items()},
}

integrity_audit

{'null_values': {'id': 0, 'model': 0, 'description': 0},
 'null_values_total': 0,
 'duplicate_ids': 0,
 'unique_ids': 6500,
 'empty_strings': {'id': 0, 'model': 0, 'description': 0}}

In [12]:
ai_groups_available = group_composition["ai_rows"].ge(1)
generator_counts = metadata_df.loc[ai_mask, "model"].value_counts().sort_index()
ideal_generator_distribution_500 = (generator_counts / generator_counts.sum() * 500).round(2)

subset_checks = {
    "exactly_500_humans": bool(human_mask.sum() == 500),
    "at_least_500_ai": bool(ai_mask.sum() >= 500),
    "at_least_500_descriptions": bool(metadata_df["description"].nunique(dropna=False) >= 500),
    "each_description_has_at_least_one_ai": bool(ai_groups_available.all()),
    "possible_one_ai_for_500_descriptions": bool(ai_groups_available.sum() >= 500),
    "approx_generator_distribution_for_500_ai": ideal_generator_distribution_500.to_dict(),
}

subset_checks

{'exactly_500_humans': True,
 'at_least_500_ai': True,
 'at_least_500_descriptions': True,
 'each_description_has_at_least_one_ai': True,
 'possible_one_ai_for_500_descriptions': True,
 'approx_generator_distribution_for_500_ai': {'AudioLDM 2 Large': 41.67,
  'AudioLDM 2 Music': 41.67,
  'MusicGen Large': 41.67,
  'MusicGen Medium': 41.67,
  'MusicGen Small': 41.67,
  'Mustango': 41.67,
  'Riffusion': 41.67,
  'Stable Audio v1': 41.67,
  'Stable Audio v2': 41.67,
  'Suno v3': 41.67,
  'Suno v3.5': 41.67,
  'Udio': 41.67}}

In [13]:
phase2_summary = {
    "metadata_rows": int(len(metadata_df)),
    "unique_ids": unique_ids,
    "unique_models": int(len(model_values)),
    "human_rows": int(human_mask.sum()),
    "ai_rows": int(ai_mask.sum()),
    "unique_descriptions": int(metadata_df["description"].nunique(dropna=False)),
    "description_min_group_size": int(description_group_sizes.min()),
    "description_max_group_size": int(description_group_sizes.max()),
    "groups_with_1_human": groups_with_1_human,
    "groups_with_12_ai": groups_with_12_ai,
    "groups_with_13_rows": groups_with_13_rows,
    "anomalous_groups": int(len(anomalous_groups_df)),
    "null_values_total": int(null_values.sum()),
    "duplicate_ids": duplicate_ids,
    "subset_500_500_feasible": bool(
        subset_checks["exactly_500_humans"]
        and subset_checks["at_least_500_ai"]
        and subset_checks["possible_one_ai_for_500_descriptions"]
    ),
}

print("RESULTADOS OBSERVADOS - FASE 2")
for key, value in phase2_summary.items():
    print(f"{key}: {value}")

RESULTADOS OBSERVADOS - FASE 2
metadata_rows: 6500
unique_ids: 6500
unique_models: 13
human_rows: 500
ai_rows: 6000
unique_descriptions: 500
description_min_group_size: 13
description_max_group_size: 13
groups_with_1_human: 500
groups_with_12_ai: 500
groups_with_13_rows: 500
anomalous_groups: 0
null_values_total: 0
duplicate_ids: 0
subset_500_500_feasible: True


### 2.1. Trazabilidad del identificador

In [14]:
survey_dataset_id = "disco-eth/AIME-survey"
survey_revision = "d7b8ee3a7aafcf2660224c8b7b8c9691e3543d59"
survey_columns = [
    "question-type",
    "description",
    "model-1",
    "track-1-id",
    "track-1-begin",
    "track-1-end",
    "model-2",
    "track-2-id",
    "track-2-begin",
    "track-2-end",
]

survey_info = api.dataset_info(survey_dataset_id, revision=survey_revision, files_metadata=False)
survey_repo_files = list(
    api.list_repo_tree(
        survey_dataset_id,
        repo_type="dataset",
        revision=survey_revision,
        path_in_repo="data",
        recursive=True,
        expand=True,
    )
)
survey_parquet_files = sorted(
    [file for file in survey_repo_files if getattr(file, "path", "").endswith(".parquet")],
    key=lambda file: file.path,
)
survey_parquet_paths = [
    f"datasets/{survey_dataset_id}@{survey_revision}/{file.path}"
    for file in survey_parquet_files
]

survey_summary = {
    "dataset_id": survey_info.id,
    "revision": survey_info.sha,
    "parquet_files": [file.path for file in survey_parquet_files],
    "remote_size_bytes": sum(getattr(file, "size", 0) or 0 for file in survey_parquet_files),
    "card_data_type": type(survey_info.cardData).__name__,
}

survey_summary

{'dataset_id': 'disco-eth/AIME-survey',
 'revision': 'd7b8ee3a7aafcf2660224c8b7b8c9691e3543d59',
 'parquet_files': ['data/train-00000-of-00001.parquet'],
 'remote_size_bytes': 295100,
 'card_data_type': 'DatasetCardData'}

In [15]:
survey_table = pq.read_table(
    survey_parquet_paths,
    filesystem=hf_fs,
    columns=survey_columns,
)
assert survey_table.column_names == survey_columns

survey_df = survey_table.to_pandas()
survey_feature_repr = {
    feature["name"]: repr(feature)
    for feature in survey_info.cardData.dataset_info["features"]
}

track_1_ids = survey_df["track-1-id"].dropna().astype(int).astype(str).str.zfill(5)
track_2_ids = survey_df["track-2-id"].dropna().astype(int).astype(str).str.zfill(5)
survey_track_ids = set(track_1_ids).union(set(track_2_ids))
aime_ids = set(metadata_df["id"].astype(str))
survey_ids_match_aime_ids = survey_track_ids.issubset(aime_ids)
missing_survey_ids = sorted(survey_track_ids - aime_ids)

survey_models_with_mtg = survey_df[
    survey_df["model-1"].eq("MTG-Jamendo") | survey_df["model-2"].eq("MTG-Jamendo")
].head(10)

explicit_original_fields = [
    field for field in survey_columns + metadata_columns
    if any(token in field.lower() for token in ["artist", "album", "song", "original", "jamendo"])
]

id_identity_status = "aime_clip_id_only" if survey_ids_match_aime_ids and not explicit_original_fields else "mapping_unclear"

identifier_traceability = {
    "survey_columns": survey_columns,
    "survey_features": survey_feature_repr,
    "survey_rows_read": int(len(survey_df)),
    "unique_track_1_ids": int(track_1_ids.nunique()),
    "unique_track_2_ids": int(track_2_ids.nunique()),
    "unique_survey_track_ids": int(len(survey_track_ids)),
    "survey_ids_match_aime_ids": bool(survey_ids_match_aime_ids),
    "missing_survey_ids_count": len(missing_survey_ids),
    "explicit_original_song_fields": explicit_original_fields,
    "id_identity_status": id_identity_status,
    "evidence": "track-1-id y track-2-id coinciden con AIME.id tras normalizar a cinco digitos; no hay campos explicitos de cancion original, artista, album ni ruta MTG-Jamendo en los datasets auditados.",
}

identifier_traceability, survey_models_with_mtg

({'survey_columns': ['question-type',
   'description',
   'model-1',
   'track-1-id',
   'track-1-begin',
   'track-1-end',
   'model-2',
   'track-2-id',
   'track-2-begin',
   'track-2-end'],
  'survey_features': {'question-type': "{'name': 'question-type', 'dtype': 'string'}",
   'description': "{'name': 'description', 'dtype': 'string'}",
   'model-1': "{'name': 'model-1', 'dtype': 'string'}",
   'track-1-id': "{'name': 'track-1-id', 'dtype': 'int64'}",
   'track-1-begin': "{'name': 'track-1-begin', 'dtype': 'string'}",
   'track-1-end': "{'name': 'track-1-end', 'dtype': 'string'}",
   'model-2': "{'name': 'model-2', 'dtype': 'string'}",
   'track-2-id': "{'name': 'track-2-id', 'dtype': 'int64'}",
   'track-2-begin': "{'name': 'track-2-begin', 'dtype': 'string'}",
   'track-2-end': "{'name': 'track-2-end', 'dtype': 'string'}",
   'answer': "{'name': 'answer', 'dtype': 'int64'}"},
  'survey_rows_read': 15600,
  'unique_track_1_ids': 1279,
  'unique_track_2_ids': 1271,
  'unique_sur

## 3. Microprueba de acceso y decodificacion de audio

In [16]:
import re
import shutil
import subprocess

selected_description = sorted(metadata_df["description"].unique())[0]
selected_group = metadata_df[metadata_df["description"].eq(selected_description)].sort_values("model")
selected_audio_rows = pd.concat([
    selected_group[selected_group["model"].eq("MTG-Jamendo")].head(1),
    selected_group[selected_group["model"].ne("MTG-Jamendo")].head(1),
])[metadata_columns].reset_index(drop=True)
selected_audio_ids = selected_audio_rows["id"].astype(str).tolist()

print("selected_description:", selected_description)
selected_audio_rows

selected_description: ambient, blues, piano


,id,model,description
0,06001,MTG-Jamendo,"ambient, blues, piano"
1,01631,AudioLDM 2 Large,"ambient, blues, piano"


In [17]:
audio_locations = []
remaining_ids = set(selected_audio_ids)

for file, parquet_path in zip(parquet_files, parquet_paths):
    if not remaining_ids:
        break
    matches = pq.read_table(
        parquet_path,
        filesystem=hf_fs,
        columns=metadata_columns,
        filters=[("description", "=", selected_description)],
    )
    if matches.num_rows == 0:
        continue

    shard_df = matches.to_pandas()
    shard_df = shard_df[shard_df["id"].astype(str).isin(remaining_ids)]
    if shard_df.empty:
        continue

    pf = pq.ParquetFile(parquet_path, filesystem=hf_fs)
    for row in shard_df.itertuples(index=False):
        row_id = str(row.id)
        for row_group_index in range(pf.num_row_groups):
            row_group_ids = [str(value) for value in pf.read_row_group(row_group_index, columns=["id"]).column("id").to_pylist()]
            if row_id in row_group_ids:
                audio_locations.append({
                    "id": row_id,
                    "model": row.model,
                    "description": row.description,
                    "shard": file.path,
                    "row_group": row_group_index,
                    "row_in_row_group": row_group_ids.index(row_id),
                    "rows_in_row_group": pf.metadata.row_group(row_group_index).num_rows,
                    "remote_shard_size_bytes": getattr(file, "size", None),
                })
                remaining_ids.remove(row_id)
                break

audio_locations_df = pd.DataFrame(audio_locations).sort_values("model").reset_index(drop=True)
same_audio_shard = audio_locations_df["shard"].nunique() == 1
involved_remote_shard_size_bytes = int(audio_locations_df.drop_duplicates("shard")["remote_shard_size_bytes"].sum())

audio_location_summary = {
    "same_audio_shard": bool(same_audio_shard),
    "involved_remote_shard_size_bytes": involved_remote_shard_size_bytes,
    "locations": audio_locations_df.to_dict(orient="records"),
}

audio_location_summary

{'same_audio_shard': False,
 'involved_remote_shard_size_bytes': 1116641072,
 'locations': [{'id': '01631',
   'model': 'AudioLDM 2 Large',
   'description': 'ambient, blues, piano',
   'shard': 'data/train-00069-of-00210.parquet',
   'row_group': 0,
   'row_in_row_group': 5,
   'rows_in_row_group': 31,
   'remote_shard_size_bytes': 19220621},
  {'id': '06001',
   'model': 'MTG-Jamendo',
   'description': 'ambient, blues, piano',
   'shard': 'data/train-00019-of-00210.parquet',
   'row_group': 0,
   'row_in_row_group': 11,
   'rows_in_row_group': 31,
   'remote_shard_size_bytes': 1097420451}]}

In [18]:
import soundfile as sf

ffprobe_path = shutil.which("ffprobe")
audio_output_dir = repo_root / "data" / "audio" / "aime_microtest"
audio_output_dir.mkdir(parents=True, exist_ok=True)

def safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")

def directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file())

def ffprobe_audio(path: Path) -> dict:
    if ffprobe_path is None:
        return {"available": False, "error": "ffprobe no esta disponible en PATH"}
    command = [
        ffprobe_path,
        "-v", "error",
        "-show_entries", "format=format_name,duration:stream=codec_name,sample_rate,channels",
        "-of", "json",
        str(path),
    ]
    try:
        completed = subprocess.run(command, check=True, capture_output=True, text=True)
        return {"available": True, "result": json.loads(completed.stdout)}
    except Exception as exc:
        return {"available": True, "error": repr(exc)}

cache_before_audio_bytes = directory_size_bytes(cache_root)
audio_start = time.perf_counter()
audio_records = []
audio_errors = []

for row in audio_locations_df.itertuples(index=False):
    parquet_path = f"datasets/{dataset_id}@{revision}/{row.shard}"
    try:
        audio_table = pq.read_table(
            parquet_path,
            filesystem=hf_fs,
            columns=["id", "model", "description", "audio"],
            filters=[("id", "=", row.id)],
        )
        audio_dict = audio_table.to_pydict()["audio"][0]
        audio_bytes = audio_dict.get("bytes")
        audio_path = audio_dict.get("path")
        bytes_available = isinstance(audio_bytes, (bytes, bytearray))

        local_path = None
        soundfile_info = None
        decoded = False
        decode_error = None
        frames = None
        ffprobe_result = None

        if bytes_available:
            local_path = audio_output_dir / f"{safe_slug(row.model)}_{row.id}.wav"
            local_path.write_bytes(audio_bytes)
            ffprobe_result = ffprobe_audio(local_path)
            try:
                soundfile_info = sf.info(str(local_path))
                with sf.SoundFile(str(local_path)) as audio_file:
                    frames = len(audio_file)
                decoded = True
            except Exception as exc:
                decode_error = repr(exc)
                audio_errors.append({"id": row.id, "error": decode_error})
        else:
            audio_errors.append({"id": row.id, "error": "audio bytes no disponibles"})

        audio_records.append({
            "id": row.id,
            "model": row.model,
            "description": row.description,
            "shard": row.shard,
            "row_group": int(row.row_group),
            "row_in_row_group": int(row.row_in_row_group),
            "remote_shard_size_bytes": int(row.remote_shard_size_bytes),
            "audio_python_type": type(audio_dict).__name__,
            "audio_keys": sorted(audio_dict.keys()),
            "audio_has_bytes": bytes_available,
            "audio_has_path": audio_path is not None,
            "audio_path": audio_path,
            "audio_bytes_size": len(audio_bytes) if bytes_available else None,
            "local_path": str(local_path) if local_path else None,
            "local_file_size_bytes": local_path.stat().st_size if local_path else None,
            "ffprobe": ffprobe_result,
            "format": soundfile_info.format if soundfile_info else None,
            "codec": soundfile_info.subtype if soundfile_info else None,
            "sample_rate": soundfile_info.samplerate if soundfile_info else None,
            "channels": soundfile_info.channels if soundfile_info else None,
            "duration": soundfile_info.duration if soundfile_info else None,
            "frames": frames,
            "decoded": decoded,
            "error": decode_error,
        })
    except Exception as exc:
        audio_errors.append({"id": row.id, "error": repr(exc)})

audio_access_seconds = time.perf_counter() - audio_start
cache_after_audio_bytes = directory_size_bytes(cache_root)
cache_delta_audio_bytes = cache_after_audio_bytes - cache_before_audio_bytes

audio_audit_df = pd.DataFrame(audio_records)
audio_audit_summary = {
    "audio_access_seconds": audio_access_seconds,
    "cache_before_audio_bytes": cache_before_audio_bytes,
    "cache_after_audio_bytes": cache_after_audio_bytes,
    "cache_delta_bytes": cache_delta_audio_bytes,
    "total_local_audio_bytes": int(audio_audit_df["local_file_size_bytes"].fillna(0).sum()) if not audio_audit_df.empty else 0,
    "ffprobe_path": ffprobe_path,
    "audio_errors": audio_errors,
}

audio_audit_summary, audio_audit_df

({'audio_access_seconds': 159.79546659998596,
  'cache_before_audio_bytes': 8149,
  'cache_after_audio_bytes': 8149,
  'cache_delta_bytes': 0,
  'total_local_audio_bytes': 49817216,
  'ffprobe_path': None,
  'audio_errors': []},
       id             model            description  \
 0  01631  AudioLDM 2 Large  ambient, blues, piano   
 1  06001       MTG-Jamendo  ambient, blues, piano   
 
                                shard  row_group  row_in_row_group  \
 0  data/train-00069-of-00210.parquet          0                 5   
 1  data/train-00019-of-00210.parquet          0                11   
 
    remote_shard_size_bytes audio_python_type     audio_keys  audio_has_bytes  \
 0                 19220621              dict  [bytes, path]             True   
 1               1097420451              dict  [bytes, path]             True   
 
    ...  local_file_size_bytes  \
 0  ...                 640058   
 1  ...               49177158   
 
                                              

In [19]:
human_audio_row = audio_audit_df[audio_audit_df["model"].eq("MTG-Jamendo")].iloc[0]
ai_audio_row = audio_audit_df[~audio_audit_df["model"].eq("MTG-Jamendo")].iloc[0]

microtest_summary = {
    "id_identity_status": id_identity_status,
    "survey_ids_match_aime_ids": bool(survey_ids_match_aime_ids),
    "selected_description": selected_description,
    "selected_audio_rows": selected_audio_rows.to_dict(orient="records"),
    "human_audio_available": bool(human_audio_row["audio_has_bytes"]),
    "ai_audio_available": bool(ai_audio_row["audio_has_bytes"]),
    "human_audio_decoded": bool(human_audio_row["decoded"]),
    "ai_audio_decoded": bool(ai_audio_row["decoded"]),
    "total_local_audio_bytes": int(audio_audit_summary["total_local_audio_bytes"]),
    "audio_access_seconds": audio_access_seconds,
    "cache_delta_bytes": int(cache_delta_audio_bytes),
    "audio_errors": audio_errors,
}

print("RESULTADOS OBSERVADOS - FASE 3")
for key, value in microtest_summary.items():
    print(f"{key}: {value}")

RESULTADOS OBSERVADOS - FASE 3
id_identity_status: aime_clip_id_only
survey_ids_match_aime_ids: True
selected_description: ambient, blues, piano
selected_audio_rows: [{'id': '06001', 'model': 'MTG-Jamendo', 'description': 'ambient, blues, piano'}, {'id': '01631', 'model': 'AudioLDM 2 Large', 'description': 'ambient, blues, piano'}]
human_audio_available: True
ai_audio_available: True
human_audio_decoded: True
ai_audio_decoded: True
total_local_audio_bytes: 49817216
audio_access_seconds: 159.79546659998596
cache_delta_bytes: 0
audio_errors: []


### 3.1. Segmentacion utilizada en AIME-survey

In [20]:
def parse_hhmmss_to_seconds(value: str) -> float:
    parts = str(value).split(":")
    if len(parts) != 3:
        return float("nan")
    hours, minutes, seconds = parts
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

track_1_intervals = survey_df[["track-1-id", "track-1-begin", "track-1-end"]].rename(
    columns={"track-1-id": "id", "track-1-begin": "begin", "track-1-end": "end"}
)
track_2_intervals = survey_df[["track-2-id", "track-2-begin", "track-2-end"]].rename(
    columns={"track-2-id": "id", "track-2-begin": "begin", "track-2-end": "end"}
)

survey_intervals = pd.concat([track_1_intervals, track_2_intervals], ignore_index=True)
survey_intervals["id"] = survey_intervals["id"].astype(int).astype(str).str.zfill(5)
survey_intervals = survey_intervals.drop_duplicates(["id", "begin", "end"]).reset_index(drop=True)
survey_intervals["begin_seconds"] = survey_intervals["begin"].map(parse_hhmmss_to_seconds)
survey_intervals["end_seconds"] = survey_intervals["end"].map(parse_hhmmss_to_seconds)
survey_intervals["interval_duration"] = survey_intervals["end_seconds"] - survey_intervals["begin_seconds"]

survey_intervals_joined = survey_intervals.merge(
    metadata_df[metadata_columns],
    on="id",
    how="left",
    validate="many_to_one",
)
missing_interval_ids = sorted(set(survey_intervals["id"]) - set(metadata_df["id"].astype(str)))
join_nulls = survey_intervals_joined[metadata_columns].isna().sum()

survey_interval_summary = {
    "survey_normalized_intervals": int(len(survey_intervals_joined)),
    "survey_unique_ids": int(survey_intervals_joined["id"].nunique()),
    "survey_unique_descriptions": int(survey_intervals_joined["description"].nunique(dropna=True)),
    "survey_models": int(survey_intervals_joined["model"].nunique(dropna=True)),
    "missing_interval_ids": missing_interval_ids,
    "join_nulls": join_nulls.to_dict(),
}

survey_interval_summary

{'survey_normalized_intervals': 1300,
 'survey_unique_ids': 1300,
 'survey_unique_descriptions': 100,
 'survey_models': 13,
 'missing_interval_ids': [],
 'join_nulls': {'id': 0, 'model': 0, 'description': 0}}

In [21]:
interval_tolerance_seconds = 1e-6
survey_intervals_joined["is_approximately_10s"] = survey_intervals_joined["interval_duration"].sub(10).abs().le(interval_tolerance_seconds)
survey_intervals_joined["begin_gt_0"] = survey_intervals_joined["begin_seconds"].gt(0)

multi_interval_ids = (
    survey_intervals_joined.groupby(["model", "id"])[["begin_seconds", "end_seconds"]]
    .nunique()
    .max(axis=1)
    .gt(1)
    .groupby("model")
    .sum()
)

interval_stats_by_model = (
    survey_intervals_joined.groupby("model", sort=True)
    .agg(
        unique_ids=("id", "nunique"),
        intervals=("id", "size"),
        begin_min=("begin_seconds", "min"),
        begin_median=("begin_seconds", "median"),
        begin_max=("begin_seconds", "max"),
        duration_min=("interval_duration", "min"),
        duration_median=("interval_duration", "median"),
        duration_max=("interval_duration", "max"),
        approximately_10s=("is_approximately_10s", "sum"),
        begin_gt_0_count=("begin_gt_0", "sum"),
    )
)
interval_stats_by_model["approximately_10s_percent"] = (
    interval_stats_by_model["approximately_10s"] / interval_stats_by_model["intervals"] * 100
)
interval_stats_by_model["begin_gt_0_percent"] = (
    interval_stats_by_model["begin_gt_0_count"] / interval_stats_by_model["intervals"] * 100
)
interval_stats_by_model["ids_with_multiple_intervals"] = multi_interval_ids.reindex(interval_stats_by_model.index, fill_value=0).astype(int)

all_intervals_approximately_10s = bool(survey_intervals_joined["is_approximately_10s"].all())
models_with_nonzero_begin = interval_stats_by_model[interval_stats_by_model["begin_gt_0_count"].gt(0)].index.tolist()
models_with_multiple_intervals_per_id = interval_stats_by_model[
    interval_stats_by_model["ids_with_multiple_intervals"].gt(0)
].index.tolist()

interval_stats_by_model

,unique_ids,intervals,begin_min,begin_median,begin_max,duration_min,duration_median,duration_max,approximately_10s,begin_gt_0_count,approximately_10s_percent,begin_gt_0_percent,ids_with_multiple_intervals
model,,,,,,,,,,,,,
AudioLDM 2 Large,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
AudioLDM 2 Music,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
MTG-Jamendo,100,100,0.0,91.5,351.0,10.0,10.0,10.0,100,98,100.0,98.0,0
MusicGen Large,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
MusicGen Medium,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
MusicGen Small,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
Mustango,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
Riffusion,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0
Stable Audio v1,100,100,0.0,0.0,0.0,10.0,10.0,10.0,100,0,100.0,0.0,0


In [22]:
covered_aime_ids = set(survey_intervals_joined["id"])
survey_coverage_percent = len(covered_aime_ids) / len(metadata_df) * 100
coverage_by_model = (
    metadata_df.assign(covered=metadata_df["id"].astype(str).isin(covered_aime_ids))
    .groupby("model", sort=True)
    .agg(aime_rows=("id", "size"), covered_ids=("covered", "sum"))
)
coverage_by_model["coverage_percent"] = coverage_by_model["covered_ids"] / coverage_by_model["aime_rows"] * 100

covered_descriptions = set(survey_intervals_joined["description"].dropna())
survey_description_coverage_percent = len(covered_descriptions) / metadata_df["description"].nunique(dropna=False) * 100
covered_description_model_distribution = (
    survey_intervals_joined.dropna(subset=["description", "model"])
    .drop_duplicates(["description", "model"])
    .groupby("description")["model"]
    .nunique()
    .value_counts()
    .sort_index()
)

if len(covered_aime_ids) == len(metadata_df) and len(covered_descriptions) == metadata_df["description"].nunique(dropna=False):
    survey_coverage_status = "survey_covers_full_aime"
elif len(covered_aime_ids) < len(metadata_df) and len(covered_descriptions) <= metadata_df["description"].nunique(dropna=False):
    survey_coverage_status = "survey_covers_subset"
else:
    survey_coverage_status = "survey_coverage_unclear"

survey_coverage_summary = {
    "covered_aime_ids": int(len(covered_aime_ids)),
    "survey_coverage_percent": float(survey_coverage_percent),
    "covered_descriptions": int(len(covered_descriptions)),
    "survey_description_coverage_percent": float(survey_description_coverage_percent),
    "covered_description_model_distribution": covered_description_model_distribution.to_dict(),
    "survey_coverage_status": survey_coverage_status,
}

survey_coverage_summary, coverage_by_model

({'covered_aime_ids': 1300,
  'survey_coverage_percent': 20.0,
  'covered_descriptions': 100,
  'survey_description_coverage_percent': 20.0,
  'covered_description_model_distribution': {13: 100},
  'survey_coverage_status': 'survey_covers_subset'},
                   aime_rows  covered_ids  coverage_percent
 model                                                     
 AudioLDM 2 Large        500          100              20.0
 AudioLDM 2 Music        500          100              20.0
 MTG-Jamendo             500          100              20.0
 MusicGen Large          500          100              20.0
 MusicGen Medium         500          100              20.0
 MusicGen Small          500          100              20.0
 Mustango                500          100              20.0
 Riffusion               500          100              20.0
 Stable Audio v1         500          100              20.0
 Stable Audio v2         500          100              20.0
 Suno v3                 500   

In [23]:
shard_model_rows = []
row_start = 0

for file, parquet_path in zip(parquet_files, parquet_paths):
    parquet_file = pq.ParquetFile(parquet_path, filesystem=hf_fs)
    shard_total_rows = parquet_file.metadata.num_rows
    shard_slice = metadata_df.iloc[row_start:row_start + shard_total_rows]
    row_start += shard_total_rows

    shard_counts = shard_slice["model"].value_counts().sort_index()
    homogeneous = len(shard_counts) == 1
    for model, rows in shard_counts.items():
        attributed_size = (getattr(file, "size", 0) or 0) * int(rows) / shard_total_rows
        shard_model_rows.append({
            "shard": file.path,
            "model": model,
            "rows": int(rows),
            "shard_rows": int(shard_total_rows),
            "remote_shard_size_bytes": int(getattr(file, "size", 0) or 0),
            "attributed_remote_size_bytes": attributed_size,
            "homogeneous_shard": homogeneous,
        })

assert row_start == len(metadata_df)
shard_model_df = pd.DataFrame(shard_model_rows)
remote_size_by_model = (
    shard_model_df.groupby("model", sort=True)
    .agg(
        rows=("rows", "sum"),
        shards=("shard", "nunique"),
        attributed_remote_size_bytes=("attributed_remote_size_bytes", "sum"),
    )
)
remote_size_by_model["approx_remote_size_per_row_bytes"] = (
    remote_size_by_model["attributed_remote_size_bytes"] / remote_size_by_model["rows"]
)
remote_size_by_model = remote_size_by_model.sort_values("approx_remote_size_per_row_bytes", ascending=False)

homogeneous_shards = int(shard_model_df.drop_duplicates("shard")["homogeneous_shard"].sum())
mixed_shards = int(shard_model_df["shard"].nunique() - homogeneous_shards)
largest_remote_model = remote_size_by_model.index[0]
smallest_remote_model = remote_size_by_model.index[-1]

remote_size_summary = {
    "method_note": "metadata_df ya contiene model leido por proyeccion columnar; se asigna a shards por orden de lectura y filas Parquet, sin leer audio.",
    "homogeneous_shards": homogeneous_shards,
    "mixed_shards": mixed_shards,
    "largest_remote_model": largest_remote_model,
    "smallest_remote_model": smallest_remote_model,
}

remote_size_summary, remote_size_by_model

({'method_note': 'metadata_df ya contiene model leido por proyeccion columnar; se asigna a shards por orden de lectura y filas Parquet, sin leer audio.',
  'homogeneous_shards': 157,
  'mixed_shards': 53,
  'largest_remote_model': 'Suno v3.5',
  'smallest_remote_model': 'AudioLDM 2 Large'},
                   rows  shards  attributed_remote_size_bytes  \
 model                                                          
 Suno v3.5          500      22                  1.986471e+10   
 MTG-Jamendo        500      42                  1.724152e+10   
 Suno v3            500      31                  1.442067e+10   
 Udio               500      20                  3.418742e+09   
 Stable Audio v1    500      17                  1.395014e+09   
 MusicGen Small     500      17                  1.053795e+09   
 Stable Audio v2    500      18                  1.034330e+09   
 MusicGen Large     500      17                  9.380107e+08   
 MusicGen Medium    500      17                  7.430670e

In [24]:
audio_path_projection_status = "audio_path_projection_not_safe"
audio_path_examples = []
audio_path_schema = None

try:
    sample_audio_path_table = pq.read_table(
        parquet_paths[0],
        filesystem=hf_fs,
        columns=["audio.path"],
    )
    audio_path_schema = str(sample_audio_path_table.schema)
    if "bytes" not in audio_path_schema and sample_audio_path_table.column_names == ["path"]:
        audio_path_projection_status = "audio_path_projection_safe"
        audio_path_examples = sample_audio_path_table.slice(0, 10).to_pydict()["path"]
except Exception as exc:
    audio_path_projection_status = "audio_path_projection_not_safe"
    audio_path_examples = [repr(exc)]

audio_path_projection = {
    "audio_path_projection_status": audio_path_projection_status,
    "audio_path_schema": audio_path_schema,
    "audio_path_examples": audio_path_examples,
}

audio_path_projection

{'audio_path_projection_status': 'audio_path_projection_safe',
 'audio_path_schema': 'path: string\n-- schema metadata --\nhuggingface: \'{"info": {"features": {"id": {"dtype": "string", "_type": "\' + 141',
 'audio_path_examples': ['ambient_blues_piano.wav',
  'ambient_breakbeat_trance.wav',
  'ambient_calm_happy.wav',
  'ambient_calm_soft.wav',
  'ambient_calm_space.wav',
  'ambient_chillout_downtempo.wav',
  'ambient_chillout_drums.wav',
  'ambient_chillout_hiphop.wav',
  'ambient_chillout_piano.wav',
  'ambient_chillout_soundtrack.wav']}

In [25]:
raw_audio_heterogeneous_in_microtest = bool(
    audio_audit_df["duration"].nunique() > 1
    or audio_audit_df["sample_rate"].nunique() > 1
    or audio_audit_df["channels"].nunique() > 1
    or audio_audit_df["codec"].nunique() > 1
)

if all_intervals_approximately_10s and raw_audio_heterogeneous_in_microtest:
    segmentation_status = "uniform_10s_strategy_requires_own_cropping"
elif all_intervals_approximately_10s and survey_coverage_status == "survey_covers_full_aime":
    segmentation_status = "uniform_10s_strategy_supported"
else:
    segmentation_status = "segmentation_strategy_unclear"

segmentation_summary = {
    "survey_normalized_intervals": int(len(survey_intervals_joined)),
    "survey_unique_ids": int(survey_intervals_joined["id"].nunique()),
    "survey_unique_descriptions": int(survey_intervals_joined["description"].nunique(dropna=True)),
    "survey_models": int(survey_intervals_joined["model"].nunique(dropna=True)),
    "survey_coverage_percent": float(survey_coverage_percent),
    "survey_description_coverage_percent": float(survey_description_coverage_percent),
    "survey_coverage_status": survey_coverage_status,
    "all_intervals_approximately_10s": all_intervals_approximately_10s,
    "models_with_nonzero_begin": models_with_nonzero_begin,
    "models_with_multiple_intervals_per_id": models_with_multiple_intervals_per_id,
    "largest_remote_model": largest_remote_model,
    "smallest_remote_model": smallest_remote_model,
    "audio_path_projection_status": audio_path_projection_status,
    "segmentation_status": segmentation_status,
}

print("RESULTADOS OBSERVADOS - FASE 4")
for key, value in segmentation_summary.items():
    print(f"{key}: {value}")

RESULTADOS OBSERVADOS - FASE 4
survey_normalized_intervals: 1300
survey_unique_ids: 1300
survey_unique_descriptions: 100
survey_models: 13
survey_coverage_percent: 20.0
survey_description_coverage_percent: 20.0
survey_coverage_status: survey_covers_subset
all_intervals_approximately_10s: True
models_with_nonzero_begin: ['MTG-Jamendo', 'Suno v3', 'Suno v3.5', 'Udio']
models_with_multiple_intervals_per_id: []
largest_remote_model: Suno v3.5
smallest_remote_model: AudioLDM 2 Large
audio_path_projection_status: audio_path_projection_safe
segmentation_status: uniform_10s_strategy_requires_own_cropping


## 5. Prueba minima del pipeline de caracteristicas

In [26]:
import hashlib
import shutil

acceptance_description = "ambient, blues, piano"
acceptance_raw_dir = repo_root / "data" / "audio" / "aime_acceptance_raw"
microtest_dir = repo_root / "data" / "audio" / "aime_microtest"
acceptance_raw_dir.mkdir(parents=True, exist_ok=True)

acceptance_rows_df = (
    metadata_df[metadata_df["description"].eq(acceptance_description)]
    .copy()
    .sort_values("model")
    .reset_index(drop=True)
)
acceptance_rows_df["label"] = acceptance_rows_df["model"].eq("MTG-Jamendo").map({True: 0, False: 1})
acceptance_rows_df = acceptance_rows_df[["id", "model", "description", "label"]]

assert len(acceptance_rows_df) == 13
assert acceptance_rows_df["model"].nunique() == 13
assert int(acceptance_rows_df["label"].eq(0).sum()) == 1
assert int(acceptance_rows_df["label"].eq(1).sum()) == 12

acceptance_rows_df

,id,model,description,label
0,01631,AudioLDM 2 Large,"ambient, blues, piano",1
1,02131,AudioLDM 2 Music,"ambient, blues, piano",1
2,06001,MTG-Jamendo,"ambient, blues, piano",0
3,01131,MusicGen Large,"ambient, blues, piano",1
4,00631,MusicGen Medium,"ambient, blues, piano",1
5,00131,MusicGen Small,"ambient, blues, piano",1
6,03001,Mustango,"ambient, blues, piano",1
7,02631,Riffusion,"ambient, blues, piano",1
8,03501,Stable Audio v1,"ambient, blues, piano",1
9,04001,Stable Audio v2,"ambient, blues, piano",1


In [27]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

acceptance_locations = []
row_start = 0
selected_ids = set(acceptance_rows_df["id"].astype(str))

for file, parquet_path in zip(parquet_files, parquet_paths):
    parquet_file = pq.ParquetFile(parquet_path, filesystem=hf_fs)
    shard_rows = parquet_file.metadata.num_rows
    shard_slice = metadata_df.iloc[row_start:row_start + shard_rows].copy()
    shard_slice["global_row"] = range(row_start, row_start + shard_rows)
    row_start += shard_rows

    shard_matches = shard_slice[shard_slice["id"].astype(str).isin(selected_ids)]
    if shard_matches.empty:
        continue

    row_group_offsets = []
    offset = 0
    for row_group_index in range(parquet_file.num_row_groups):
        row_group_rows = parquet_file.metadata.row_group(row_group_index).num_rows
        row_group_offsets.append((row_group_index, offset, offset + row_group_rows))
        offset += row_group_rows

    for row in shard_matches.itertuples(index=False):
        row_in_shard = int(row.global_row - (row_start - shard_rows))
        row_group = next(index for index, start, end in row_group_offsets if start <= row_in_shard < end)
        row_group_start = next(start for index, start, end in row_group_offsets if index == row_group)
        acceptance_locations.append({
            "id": str(row.id),
            "model": row.model,
            "description": row.description,
            "label": 0 if row.model == "MTG-Jamendo" else 1,
            "shard": file.path,
            "row_group": row_group,
            "row_in_row_group": row_in_shard - row_group_start,
            "remote_shard_size_bytes": int(getattr(file, "size", 0) or 0),
        })

assert row_start == len(metadata_df)
acceptance_locations_df = pd.DataFrame(acceptance_locations).sort_values("model").reset_index(drop=True)
assert len(acceptance_locations_df) == 13
acceptance_locations_df

,id,model,description,label,shard,row_group,row_in_row_group,remote_shard_size_bytes
0,01631,AudioLDM 2 Large,"ambient, blues, piano",1,data/train-00069-of-00210.parquet,0,5,19220621
1,02131,AudioLDM 2 Music,"ambient, blues, piano",1,data/train-00089-of-00210.parquet,0,23,19071751
2,06001,MTG-Jamendo,"ambient, blues, piano",0,data/train-00019-of-00210.parquet,0,11,1097420451
3,01131,MusicGen Large,"ambient, blues, piano",1,data/train-00048-of-00210.parquet,0,18,40401510
4,00631,MusicGen Medium,"ambient, blues, piano",1,data/train-00028-of-00210.parquet,0,0,40401517
5,00131,MusicGen Small,"ambient, blues, piano",1,data/train-00007-of-00210.parquet,0,13,40401537
6,03001,Mustango,"ambient, blues, piano",1,data/train-00120-of-00210.parquet,0,8,189687786
7,02631,Riffusion,"ambient, blues, piano",1,data/train-00107-of-00210.parquet,0,3,27977759
8,03501,Stable Audio v1,"ambient, blues, piano",1,data/train-00137-of-00210.parquet,0,19,458499032
9,04001,Stable Audio v2,"ambient, blues, piano",1,data/train-00154-of-00210.parquet,0,30,871063944


In [28]:
def safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")

def acceptance_filename(model: str, audio_id: str) -> str:
    return f"{safe_slug(model)}_{audio_id}.wav"

reused_audio_files = []
moved_audio_files = []

for row in acceptance_rows_df.itertuples(index=False):
    file_name = acceptance_filename(row.model, row.id)
    source_path = microtest_dir / file_name
    target_path = acceptance_raw_dir / file_name
    if source_path.exists():
        source_sha = file_sha256(source_path)
        if target_path.exists():
            target_sha = file_sha256(target_path)
            if source_sha != target_sha:
                raise ValueError(f"Conflicto de SHA-256 para {file_name}")
            source_path.unlink()
            move_action = "removed_duplicate_source"
        else:
            shutil.move(str(source_path), str(target_path))
            target_sha = file_sha256(target_path)
            if source_sha != target_sha:
                raise ValueError(f"SHA-256 distinto tras mover {file_name}")
            move_action = "moved"
        reused_audio_files.append(file_name)
        moved_audio_files.append({
            "file": file_name,
            "source_sha256": source_sha,
            "target_sha256": target_sha,
            "action": move_action,
        })

moved_audio_files

[{'file': 'audioldm-2-large_01631.wav',
  'source_sha256': '3b8af73b33425979eea5b7ccec01fd55c05017629ba94ed2ef0386c91b8c6e8c',
  'target_sha256': '3b8af73b33425979eea5b7ccec01fd55c05017629ba94ed2ef0386c91b8c6e8c',
  'action': 'moved'},
 {'file': 'mtg-jamendo_06001.wav',
  'source_sha256': 'e7fa28f19eba20d67e95d5c15b0abfd6b9fe85fef9866d1b047f90642f21f4f2',
  'target_sha256': 'e7fa28f19eba20d67e95d5c15b0abfd6b9fe85fef9866d1b047f90642f21f4f2',
  'action': 'moved'}]

In [29]:
import soundfile as sf

raw_audio_records = []
raw_audio_errors = []
newly_downloaded_audio_files = []

for row in acceptance_locations_df.itertuples(index=False):
    file_name = acceptance_filename(row.model, row.id)
    local_path = acceptance_raw_dir / file_name
    error = None
    download_seconds = 0.0

    if not local_path.exists():
        start = time.perf_counter()
        try:
            parquet_path = f"datasets/{dataset_id}@{revision}/{row.shard}"
            audio_table = pq.read_table(
                parquet_path,
                filesystem=hf_fs,
                columns=["id", "audio"],
                filters=[("id", "=", row.id)],
            )
            audio_column = audio_table.column("audio")
            audio_value = audio_column[0].as_py()
            audio_bytes = audio_value.get("bytes") if isinstance(audio_value, dict) else None
            if not isinstance(audio_bytes, (bytes, bytearray)):
                raise ValueError("audio.bytes no disponible")
            local_path.write_bytes(audio_bytes)
            newly_downloaded_audio_files.append(file_name)
        except Exception as exc:
            error = repr(exc)
            raw_audio_errors.append({"id": row.id, "model": row.model, "error": error})
        finally:
            download_seconds = time.perf_counter() - start

    decoded = False
    format_name = subtype = sample_rate = channels = duration_seconds = frames = file_size_bytes = sha256 = None
    if local_path.exists():
        try:
            info = sf.info(str(local_path))
            with sf.SoundFile(str(local_path)) as audio_file:
                frames = len(audio_file)
            decoded = True
            format_name = info.format
            subtype = info.subtype
            sample_rate = info.samplerate
            channels = info.channels
            duration_seconds = info.duration
            file_size_bytes = local_path.stat().st_size
            sha256 = file_sha256(local_path)
        except Exception as exc:
            error = repr(exc)
            raw_audio_errors.append({"id": row.id, "model": row.model, "error": error})

    raw_audio_records.append({
        "id": row.id,
        "model": row.model,
        "label": int(row.label),
        "local_path": str(local_path),
        "sha256": sha256,
        "file_size_bytes": file_size_bytes,
        "format": format_name,
        "subtype": subtype,
        "sample_rate": sample_rate,
        "channels": channels,
        "duration_seconds": duration_seconds,
        "frames": frames,
        "decoded": decoded,
        "download_seconds": download_seconds,
        "error": error,
    })

raw_audio_df = pd.DataFrame(raw_audio_records).sort_values("model").reset_index(drop=True)
assert len(raw_audio_df) == 13
raw_audio_df

,id,model,label,local_path,sha256,file_size_bytes,format,subtype,sample_rate,channels,duration_seconds,frames,decoded,download_seconds,error
0,01631,AudioLDM 2 Large,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,3b8af73b33425979eea5b7ccec01fd55c05017629ba94e...,640058,WAV,FLOAT,16000,1,10.000000,160000,True,0.000000,None
1,02131,AudioLDM 2 Music,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,1c05a93f5e1a1d79ecd58dd531bf4969ecbabf9798e349...,640058,WAV,FLOAT,16000,1,10.000000,160000,True,6.687163,None
2,06001,MTG-Jamendo,0,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,e7fa28f19eba20d67e95d5c15b0abfd6b9fe85fef9866d...,49177158,WAV,PCM_16,48000,2,256.130625,12294270,True,0.000000,None
3,01131,MusicGen Large,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,688af020ecac16f9e7c2f16a0a548b8ad966347fbcbe26...,1303098,WAV,FLOAT,32000,1,10.180000,325760,True,7.396584,None
4,00631,MusicGen Medium,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,0b27b0d35495f2497b6287f75c009f4f9726f261531358...,1303098,WAV,FLOAT,32000,1,10.180000,325760,True,6.869406,None
5,00131,MusicGen Small,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,a94d245ad4b954a148819f4d2e7e7c1ce53a56efddf87c...,1303098,WAV,FLOAT,32000,1,10.180000,325760,True,8.502955,None
6,03001,Mustango,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,a531b12376fade7e1362402e24cdc8d8711a6ee199e2cd...,327788,WAV,PCM_16,16000,1,10.242000,163872,True,27.107566,None
7,02631,Riffusion,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,abe7fc43d88e94543412734b842e73cff89b4b66861d70...,902364,WAV,PCM_16,44100,1,10.230000,451143,True,6.539209,None
8,03501,Stable Audio v1,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,610058d22f79181b14347aaee54dcfcfab4d0ea488694b...,1920078,WAV,PCM_16,48000,2,10.000000,480000,True,53.959113,None
9,04001,Stable Audio v2,1,C:\Users\sergio\tfm\tfm-ai-music-detection\dat...,aef446f6907bc2f569eea7dbd3d99d8b039887e934d140...,1920078,WAV,PCM_16,48000,2,10.000000,480000,True,109.585150,None


In [30]:
import librosa

def extract_10s_mono_resampled(
    path: str | Path,
    target_duration_seconds: float = 10.0,
    target_sample_rate: int = 16000,
    energy_hop_seconds: float = 0.5,
) -> dict:
    try:
        audio, source_sample_rate = sf.read(str(path), always_2d=True, dtype="float32")
        if audio.size == 0:
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": None, "duration_seconds": None, "error": "audio vacio"}

        mono = audio.mean(axis=1).astype(np.float32)
        source_duration = len(mono) / source_sample_rate
        target_source_samples = int(round(target_duration_seconds * source_sample_rate))
        if source_duration + 1e-6 < target_duration_seconds:
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": None, "duration_seconds": source_duration, "error": "audio inferior a 10 segundos"}

        if len(mono) > target_source_samples:
            hop_samples = max(1, int(round(energy_hop_seconds * source_sample_rate)))
            starts = np.arange(0, len(mono) - target_source_samples + 1, hop_samples)
            if starts[-1] != len(mono) - target_source_samples:
                starts = np.append(starts, len(mono) - target_source_samples)
            energies = np.array([
                float(np.mean(np.square(mono[start:start + target_source_samples])))
                for start in starts
            ])
            best_start = int(starts[int(np.argmax(energies))])
        else:
            best_start = 0

        segment = mono[best_start:best_start + target_source_samples]
        if source_sample_rate != target_sample_rate:
            segment = librosa.resample(segment, orig_sr=source_sample_rate, target_sr=target_sample_rate).astype(np.float32)
        else:
            segment = segment.astype(np.float32)

        target_samples = int(round(target_duration_seconds * target_sample_rate))
        length_delta = len(segment) - target_samples
        if abs(length_delta) <= 2:
            if length_delta > 0:
                segment = segment[:target_samples]
            elif length_delta < 0:
                segment = np.pad(segment, (0, -length_delta))
        elif len(segment) != target_samples:
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": best_start / source_sample_rate, "duration_seconds": len(segment) / target_sample_rate, "error": f"longitud inesperada: {len(segment)}"}

        if not np.all(np.isfinite(segment)):
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": best_start / source_sample_rate, "duration_seconds": target_duration_seconds, "error": "segmento no finito"}

        return {
            "waveform": segment.astype(np.float32),
            "sample_rate": target_sample_rate,
            "start_seconds": best_start / source_sample_rate,
            "duration_seconds": len(segment) / target_sample_rate,
            "error": None,
        }
    except Exception as exc:
        return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": None, "duration_seconds": None, "error": repr(exc)}

preprocess_records = []
preprocessed_waveforms = []
target_sample_rate = 16000
target_duration_seconds = 10.0
target_num_samples = int(target_sample_rate * target_duration_seconds)

for row in raw_audio_df.itertuples(index=False):
    result = extract_10s_mono_resampled(row.local_path, target_duration_seconds=target_duration_seconds, target_sample_rate=target_sample_rate)
    waveform = result["waveform"]
    valid = waveform is not None and waveform.ndim == 1 and len(waveform) == target_num_samples and np.all(np.isfinite(waveform))
    if valid:
        preprocessed_waveforms.append(waveform)
    preprocess_records.append({
        "id": row.id,
        "model": row.model,
        "label": int(row.label),
        "start_seconds": result["start_seconds"],
        "final_sample_rate": result["sample_rate"],
        "num_samples": len(waveform) if waveform is not None else None,
        "duration_seconds": result["duration_seconds"],
        "valid": bool(valid),
        "error": result["error"],
    })

preprocess_df = pd.DataFrame(preprocess_records)
preprocess_errors = preprocess_df[preprocess_df["error"].notna()].to_dict(orient="records")
assert int(preprocess_df["valid"].sum()) == 13
waveforms = np.stack(preprocessed_waveforms).astype(np.float32)
preprocess_df

,id,model,label,start_seconds,final_sample_rate,num_samples,duration_seconds,valid,error
0,01631,AudioLDM 2 Large,1,0.00,16000,160000,10.0,True,None
1,02131,AudioLDM 2 Music,1,0.00,16000,160000,10.0,True,None
2,06001,MTG-Jamendo,0,78.00,16000,160000,10.0,True,None
3,01131,MusicGen Large,1,0.00,16000,160000,10.0,True,None
4,00631,MusicGen Medium,1,0.00,16000,160000,10.0,True,None
5,00131,MusicGen Small,1,0.18,16000,160000,10.0,True,None
6,03001,Mustango,1,0.00,16000,160000,10.0,True,None
7,02631,Riffusion,1,0.00,16000,160000,10.0,True,None
8,03501,Stable Audio v1,1,0.00,16000,160000,10.0,True,None
9,04001,Stable Audio v2,1,0.00,16000,160000,10.0,True,None


In [31]:
mfcc_start = time.perf_counter()
mfcc_features = []
for waveform in waveforms:
    mfcc = librosa.feature.mfcc(y=waveform, sr=target_sample_rate, n_mfcc=20)
    feature_vector = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)]).astype(np.float32)
    mfcc_features.append(feature_vector)
X_mfcc = np.stack(mfcc_features)
mfcc_seconds_total = time.perf_counter() - mfcc_start
mfcc_seconds_per_audio = mfcc_seconds_total / len(X_mfcc)
mfcc_all_finite = bool(np.all(np.isfinite(X_mfcc)))
assert X_mfcc.shape == (13, 40)
assert mfcc_all_finite

mfcc_summary = {
    "mfcc_shape": X_mfcc.shape,
    "mfcc_all_finite": mfcc_all_finite,
    "mfcc_seconds_total": mfcc_seconds_total,
    "mfcc_seconds_per_audio": mfcc_seconds_per_audio,
}

mfcc_summary

{'mfcc_shape': (13, 40),
 'mfcc_all_finite': True,
 'mfcc_seconds_total': 3.1339383001904935,
 'mfcc_seconds_per_audio': 0.24107217693773025}

In [32]:
embedding_model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
embedding_device = "cpu"
embedding_error = None
embedding_seconds_total = None
embedding_seconds_per_audio = None
embedding_all_finite = False
X_embed = None
embedding_memory_bytes = None

embedding_start = time.perf_counter()
try:
    import torch
    from transformers import AutoModel, AutoProcessor

    embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(embedding_model_id)
    embedding_model = AutoModel.from_pretrained(embedding_model_id).to(embedding_device)
    embedding_model.eval()

    embeddings = []
    with torch.no_grad():
        for waveform in waveforms:
            inputs = processor(waveform, sampling_rate=target_sample_rate, return_tensors="pt")
            inputs = {key: value.to(embedding_device) for key, value in inputs.items()}
            outputs = embedding_model(**inputs)
            if hasattr(outputs, "last_hidden_state") and outputs.last_hidden_state is not None:
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
                embedding = outputs.pooler_output.squeeze(0)
            else:
                raise ValueError("el modelo no devuelve last_hidden_state ni pooler_output")
            embeddings.append(embedding.detach().cpu().numpy().astype(np.float32))

    X_embed = np.stack(embeddings)
    embedding_all_finite = bool(np.all(np.isfinite(X_embed)))
    embedding_memory_bytes = int(X_embed.nbytes)
except Exception as exc:
    embedding_error = repr(exc)
finally:
    embedding_seconds_total = time.perf_counter() - embedding_start
    embedding_seconds_per_audio = embedding_seconds_total / len(waveforms)

embedding_summary = {
    "embedding_model_id": embedding_model_id,
    "embedding_device": embedding_device,
    "embedding_shape": None if X_embed is None else X_embed.shape,
    "embedding_all_finite": embedding_all_finite,
    "embedding_seconds_total": embedding_seconds_total,
    "embedding_seconds_per_audio": embedding_seconds_per_audio,
    "embedding_memory_bytes": embedding_memory_bytes,
    "embedding_error": embedding_error,
}

embedding_summary

C:\Users\sergio\tfm\tfm-ai-music-detection\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sergio\tfm\tfm-ai-music-detection\.cache\huggingface\hub\models--MIT--ast-finetuned-audioset-10-10-0.4593. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\sergio\tfm\tfm-ai-music-detection\.venv\Lib\site-packages\transformers\audio_utils.py:704: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3520.33it/s]


[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'embedding_model_id': 'MIT/ast-finetuned-audioset-10-10-0.4593',
 'embedding_device': 'cpu',
 'embedding_shape': (13, 768),
 'embedding_all_finite': True,
 'embedding_seconds_total': 80.45121729979292,
 'embedding_seconds_per_audio': 6.188555176907148,
 'embedding_memory_bytes': 39936,
 'embedding_error': None}

In [33]:
decoded_audio_files = int(raw_audio_df["decoded"].sum())
preprocessed_segments = int(preprocess_df["valid"].sum())
embedding_shape = None if X_embed is None else X_embed.shape

if (
    len(acceptance_rows_df) == 13
    and acceptance_rows_df["model"].nunique() == 13
    and decoded_audio_files == 13
    and preprocessed_segments == 13
    and X_mfcc.shape == (13, 40)
    and mfcc_all_finite
    and X_embed is not None
    and X_embed.shape[0] == 13
    and embedding_all_finite
):
    acceptance_status = "aime_apto_con_condiciones"
elif (
    len(acceptance_rows_df) == 13
    and acceptance_rows_df["model"].nunique() == 13
    and decoded_audio_files == 13
    and preprocessed_segments == 13
    and X_mfcc.shape == (13, 40)
    and mfcc_all_finite
    and embedding_error is not None
):
    acceptance_status = "aime_apto_con_condiciones_pendiente_embedding"
else:
    acceptance_status = "aime_no_confirmado"

acceptance_summary = {
    "acceptance_rows": int(len(acceptance_rows_df)),
    "human_rows": int(acceptance_rows_df["label"].eq(0).sum()),
    "ai_rows": int(acceptance_rows_df["label"].eq(1).sum()),
    "unique_models": int(acceptance_rows_df["model"].nunique()),
    "decoded_audio_files": decoded_audio_files,
    "raw_audio_errors": raw_audio_errors,
    "preprocessed_segments": preprocessed_segments,
    "preprocess_errors": preprocess_errors,
    "target_sample_rate": target_sample_rate,
    "target_num_samples": target_num_samples,
    "mfcc_shape": X_mfcc.shape,
    "mfcc_all_finite": mfcc_all_finite,
    "mfcc_seconds_total": mfcc_seconds_total,
    "embedding_model_id": embedding_model_id,
    "embedding_shape": embedding_shape,
    "embedding_all_finite": embedding_all_finite,
    "embedding_seconds_total": embedding_seconds_total,
    "embedding_device": embedding_device,
    "acceptance_status": acceptance_status,
}

print("RESULTADOS OBSERVADOS - FASE 5")
for key, value in acceptance_summary.items():
    print(f"{key}: {value}")

RESULTADOS OBSERVADOS - FASE 5
acceptance_rows: 13
human_rows: 1
ai_rows: 12
unique_models: 13
decoded_audio_files: 13
raw_audio_errors: []
preprocessed_segments: 13
preprocess_errors: []
target_sample_rate: 16000
target_num_samples: 160000
mfcc_shape: (13, 40)
mfcc_all_finite: True
mfcc_seconds_total: 3.1339383001904935
embedding_model_id: MIT/ast-finetuned-audioset-10-10-0.4593
embedding_shape: (13, 768)
embedding_all_finite: True
embedding_seconds_total: 80.45121729979292
embedding_device: cpu
acceptance_status: aime_apto_con_condiciones
